In [1]:
! pip install datasets evaluate torchcodec

In [2]:
from datasets import load_dataset, DatasetDict

carib_voices = DatasetDict()

carib_voices["train"] = load_dataset("neddamj/carib_voices_data", split="train")
carib_voices["val"] = load_dataset("neddamj/carib_voices_data", split="val")
carib_voices["test"] = load_dataset("neddamj/carib_voices_data", split="test")

print(carib_voices)

DatasetDict({
    train: Dataset({
        features: ['ID', 'Transcription', 'audio'],
        num_rows: 18863
    })
    val: Dataset({
        features: ['ID', 'Transcription', 'audio'],
        num_rows: 993
    })
    test: Dataset({
        features: ['ID', 'Transcription', 'audio'],
        num_rows: 8510
    })
})


Load the feature extractor, tokenizer and processor

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-large-v3")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-large-v3", language="English", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3", language="English", task="transcribe")

In [7]:
from datasets import Audio

carib_voices = carib_voices.cast_column("audio", Audio(sampling_rate=16000))

In [8]:
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["Transcription"]).input_ids
    return batch

carib_voices = carib_voices.map(prepare_dataset, num_proc=2)

Map (num_proc=2):   0%|          | 0/18863 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/993 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/8510 [00:00<?, ? examples/s]

In [ ]:
from transformers import WhisperForConditionalGeneration

finetuned = True
if not finetuned:
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3")
else:
    model = WhisperForConditionalGeneration.from_pretrained("neddamj/whisper-large-carib")

model.generation_config.language = "english"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Generate the eval predictions

In [ ]:
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)
model.eval()

predictions = []

# Use a DataLoader for batch processing to speed up generation
batch_size = 16
eval_dataloader = DataLoader(carib_voices["test"], batch_size=batch_size, collate_fn=lambda x: x)

print(f"[INFO] Generating predictions on the test set (Batch size: {batch_size})")

for batch in tqdm(eval_dataloader):
    # Extract input features and stack them into a batch tensor
    input_features = torch.stack([torch.tensor(item["input_features"]) for item in batch]).to(device)

    with torch.no_grad():
        pred_ids = model.generate(input_features=input_features)
    batch_pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)
    predictions.extend(batch_pred_text)

print("Number of predictions:", len(predictions))

[INFO] Generating predictions on the test set (Batch size: 16)


100%|██████████| 532/532 [1:07:47<00:00,  7.65s/it]

Number of predictions: 8510


In [25]:
import pandas as pd

submission_df = pd.DataFrame({
    "ID": list(carib_voices["test"]["ID"]),
    "Transcription": predictions
})

submission_path = "JAIA_Carib_Voices_Submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Saved submission file to: {submission_path}")
print(submission_df.head())

Saved submission file to: SampleSubmission.csv
          ID                                      Transcription
0  ID_PDZCPT  Trinidad and Tobago set for a seventies-style ...
1  ID_ALVRZY  CARICOM's point man on bananas, Dominica Prime...
2  ID_IADSIT  Lucia's foreign minister blames the US for unl...
3  ID_WXUUAN  What plans are there for a full-scale evacuati...
4  ID_NFFJZC  Mediation talks involving the leading business...
